# Notebook 4: Layout & Navigation

This notebook covers how nodes are positioned in the 2D view and how to navigate the graph.

## Prerequisites
- Notebook 3 completed (model and connection set up)
- Neo4j running with the example DSG loaded

## 1. Spatial Layout

The graph view uses a **spatial layout** — nodes are positioned based on their actual 3D coordinates from the scene graph, projected onto the x-y plane. This means nodes that are near each other in the real environment appear near each other in the view.

The projection is: `scene_x = world_x * scale`, `scene_y = -world_y * scale` (y is negated so "north" is up, matching map conventions). The scale factor converts from meters to pixels.

The layout operates on the model's cache (plain dicts), not on spark_dsg objects.

In [ ]:
import os

from heracles.utils import get_labelspace
from PySide6.QtWidgets import QApplication

from sget.backend.scene_graph_model import SceneGraphModel
from sget.utils.layout import DEFAULT_SCALE, compute_layout

app = QApplication.instance() or QApplication([])

# Connect and load (same setup as Notebook 3).
model = SceneGraphModel()
model.connect("neo4j://127.0.0.1:7687", "neo4j", "neo4j_pw")
object_labels = get_labelspace("ade20k_mit_label_space.yaml")
room_labels = get_labelspace("b45_label_space.yaml")
model.set_labelspaces(
    object_labels={v: k for k, v in object_labels.items()},
    room_labels={v: k for k, v in room_labels.items()},
)
DSG_PATH = os.path.expanduser(
    "~/software/mit/sget/heracles/heracles/examples/scene_graphs/example_dsg.json"
)
model.load_from_json(DSG_PATH)

# Compute layout positions from the model cache.
nodes = model.get_all_nodes()
node_layers = {ns: model.get_node_layer(ns) for ns in nodes}
edges = model.get_edges()

positions = compute_layout(nodes, node_layers, edges)

print(f"Computed positions for {len(positions)} nodes")
print(f"Scale factor: {DEFAULT_SCALE} pixels per world unit\n")

# Verify the projection on a sample node.
sample_ns = list(positions.keys())[0]
sample_props = nodes[sample_ns]
scene_x, scene_y = positions[sample_ns]
wc = sample_props["center"]
print(f"Node {sample_ns}:")
print(f"  World pos:  [{wc[0]:.2f}, {wc[1]:.2f}, {wc[2]:.2f}]")
print(f"  Scene pos:  ({scene_x:.1f}, {scene_y:.1f})")
expected_x = wc[0] * DEFAULT_SCALE
expected_y = -wc[1] * DEFAULT_SCALE
print(f"  Expected:   ({expected_x:.1f}, {expected_y:.1f})")

## 2. Navigation

The graph view (`_ZoomableGraphicsView`) provides these navigation controls:

| Action | How |
|--------|-----|
| **Pan** | Click and drag (default mode) |
| **Zoom** | Mouse wheel |
| **Fit to view** | Press `F` |
| **Rubber-band select** | Hold `Shift` + drag |
| **Click select** | Click a node |
| **Multi-select** | `Ctrl` + click |
| **Search/filter** | Type in the search bar above the graph |

The default drag mode is `ScrollHandDrag` (pan). Holding Shift temporarily switches to `RubberBandDrag` for area selection. The cursor stays as an arrow (overriding Qt's default grab icon for `ScrollHandDrag`).

## 3. Search & Filter

The search bar at the top of the graph view filters nodes in real time:
- Type a string → nodes whose symbol, class, or name match stay at full opacity; non-matches dim to 15%
- Press **Enter** → all matching nodes become selected
- Clear the field → all nodes restored to full opacity

The search bar uses `ClickFocus` so it doesn't steal focus on startup — keyboard shortcuts like `F` (fit) work immediately without clicking the graph first.

In [ ]:
# Clean up.
model.disconnect()
print("Done!")

## Summary

You've learned:
- The **spatial layout** projects nodes onto the x-y plane using `(world_x * scale, -world_y * scale)`
- **Navigation**: click-drag to pan, mouse wheel to zoom, `F` to fit, Shift+drag for rubber-band
- **Search**: real-time filtering by symbol/class/name, Enter to select matches

**Next notebook**: [05_gui_and_interaction.ipynb](05_gui_and_interaction.ipynb) — the widget tree, interaction modes, and how selection flows between views.